# **ConSurf-AI**: per-residue evolutionary conservation for one or many sequences

ConSurf-AI predicts ConSurf conservation grades (**1** = variable … **9** = conserved) for every residue of a protein, without building a multiple sequence alignment. It runs a small attention network on **ESM2-3B** embeddings and only needs the **amino-acid sequence**.

This notebook takes **one sequence** or a **FASTA file with one or many sequences** and returns **tables** of grades (per residue and per sequence).

> **Want the grades shown on a 3D structure?** Use the **[ConSurf-AI structure notebook](https://colab.research.google.com/github/claudelelemaystdenis/ConSurf-AI/blob/main/colab/ConSurf_AI.ipynb)** instead: PDB ID, UniProt accession (AlphaFold DB model) or your own structure file → grades on the 3D structure, plus PyMOL / ChimeraX scripts.
>
> **Want to run ConSurf-AI on your own computer or cluster** (e.g. for large FASTA files)? The code, the model weights and a command-line tool are on **[GitHub](https://github.com/claudelelemaystdenis/ConSurf-AI)**.

**How to use**
1. `Runtime` → `Change runtime type` → select a **GPU** (T4 is enough; recommended: without a GPU, the notebook runs on the CPU, but much more slowly).
2. Run the cells in order (or `Runtime` → `Run all`). Cell **0** installs the packages and downloads the model weights once (ESM2-3B is ~11 GB, so allow a few minutes). In cell **1**, paste a sequence or tick `upload_fasta`. You can change it and rerun cells 1–3 without rerunning cell 0.
3. Speed: about **0.5 s per sequence** on a T4 once the model is loaded (~10 min for 1,000 average-length proteins). Sequences of up to **2000** residues are supported.

**Colour key.** These are the standard ConSurf colours. Yellow marks residues the model is uncertain about (predicted uncertainty ≥ 2). This plays the same role as ConSurf's "insufficient data" flag.

In [ ]:
#@title 0. Install & initialise (run first) {display-mode: "form"}
#@markdown Defines the ConSurf-AI helpers and downloads the model weights (ConSurf-AI and ESM2-3B, ~11 GB). The first run takes a few minutes.

import os, sys, re, io, json, shutil, zipfile, datetime, contextlib, importlib, subprocess
import urllib.request, urllib.error, html as _html

IN_COLAB = "google.colab" in sys.modules


def _ensure(module, pip_name=None):
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True)
        importlib.invalidate_caches()
        return importlib.import_module(module)


for _mod, _pip_name in [("transformers", "transformers"), ("huggingface_hub", "huggingface_hub")]:
    _ensure(_mod, _pip_name)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from IPython.display import display, HTML

# ── Configuration ────────────────────────────────────────────────────────────
WEIGHTS_URL = "https://raw.githubusercontent.com/claudelelemaystdenis/ConSurf-AI/main/weights/consurf_ai_3B_weights.pth"


WEIGHTS_SHA256 = "99c06aa4494a361ecfc1ec306d32109bef1d9b80d1b78be70c64d046ff15205f"


ESM2_NAME = "facebook/esm2_t36_3B_UR50D"


MAX_LEN = 2000                       # the model's rotary-embedding table covers 2000 positions


WORK_DIR = "/content/consurf_ai" if IN_COLAB else os.path.abspath(os.environ.get("CONSURF_AI_WORKDIR", "consurf_ai_run"))


WEIGHTS_PATH = os.path.join(WORK_DIR, "weights", "consurf_ai_3B_weights.pth")
os.makedirs(WORK_DIR, exist_ok=True)

CONSURF_COLORS = {1: "#10C8D1", 2: "#8CFFFF", 3: "#D7FFFF", 4: "#EAFFFF", 5: "#FFFFFF",
                  6: "#FCEDF4", 7: "#FAC9DE", 8: "#F07DAB", 9: "#A02560"}


LOWCONF_COLOR = "#FFFF96"

def _open_url(url, timeout=300):
    req = urllib.request.Request(url, headers={"User-Agent": "ConSurf-AI-Colab"})
    return urllib.request.urlopen(req, timeout=timeout)


def _download(url, path):
    tmp = path + ".part"
    with _open_url(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    os.replace(tmp, path)


# ── ConSurf-AI network (inference-only copy of step1.py's AttentionNetwork) ──
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_len=MAX_LEN):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.einsum("i,j->ij", torch.arange(max_len).float(), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :])
        self.register_buffer("sin", emb.sin()[None, None, :, :])

    def forward(self, q, k):
        L = q.size(-2)
        cos, sin = self.cos[:, :, :L, :], self.sin[:, :, :L, :]

        def rotate_half(x):
            x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
            return torch.cat((-x2, x1), dim=-1)

        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg["reduced_dim"], cfg["hidden_dimension"], bias=cfg["bias"])
        self.gelu = nn.GELU(approximate="none")
        self.c_proj = nn.Linear(cfg["hidden_dimension"], cfg["reduced_dim"], bias=cfg["bias"])

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class AttentionLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg["reduced_dim"]
        self.num_heads = cfg["num_heads"]
        self.mlp = MLP(cfg)
        self.ln_1 = nn.LayerNorm(D, bias=cfg["bias"])
        self.ln_2 = nn.LayerNorm(D, bias=cfg["bias"])
        self.rope = RotaryPositionalEmbedding(D // self.num_heads, max_len=MAX_LEN)
        self.q_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.k_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.v_proj = nn.Linear(D, D, bias=cfg["bias"])
        self.out_proj = nn.Linear(D, D, bias=cfg["bias"])

    def forward(self, x, mask):
        h = self.ln_1(x)
        B, L, D = h.shape
        H = self.num_heads
        q = self.q_proj(h).view(B, L, H, D // H).transpose(1, 2)
        k = self.k_proj(h).view(B, L, H, D // H).transpose(1, 2)
        v = self.v_proj(h).view(B, L, H, D // H).transpose(1, 2)
        q, k = self.rope(q, k)
        attn_mask = mask.unsqueeze(1).unsqueeze(2) if mask is not None else None
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=0.0, is_causal=False)
        x = x + self.out_proj(out.transpose(1, 2).contiguous().view(B, L, D))
        return x + self.mlp(self.ln_2(x))


class AttentionNetwork(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.input_reduction = nn.Linear(cfg["inferred_input_dim"], cfg["reduced_dim"], bias=cfg["bias"])
        self.layers = nn.ModuleList([AttentionLayer(cfg) for _ in range(cfg["num_layers"])])
        self.ln_f = nn.LayerNorm(cfg["reduced_dim"], bias=cfg["bias"])
        self.lm_head = nn.Linear(cfg["reduced_dim"], 9, bias=cfg["bias"])
        self.conf_head = nn.Linear(cfg["reduced_dim"], 1, bias=cfg["bias"])

    def forward(self, x, mask):
        x = self.input_reduction(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.ln_f(x)
        return self.lm_head(x), 8.0 * torch.sigmoid(self.conf_head(x).squeeze(-1))


# ── Weights download (cell 0) and model loading (cell 3, cached) ─────────────
def _sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def download_consurf_ai_weights():
    if os.path.exists(WEIGHTS_PATH):
        return WEIGHTS_PATH
    os.makedirs(os.path.dirname(WEIGHTS_PATH), exist_ok=True)
    if WEIGHTS_URL and "<" not in WEIGHTS_URL:
        print("Downloading ConSurf-AI weights...")
        for attempt in range(1, 5):
            try:
                _download(WEIGHTS_URL, WEIGHTS_PATH)
                break
            except (urllib.error.URLError, OSError) as e:   # temporary GitHub/network error (e.g. HTTP 504)
                if attempt == 4:
                    raise RuntimeError(f"Could not download the ConSurf-AI weights from {WEIGHTS_URL} ({e}). "
                                       "Please wait a few minutes and rerun cell 0.") from None
                print(f"   Download failed ({e}); retrying in {10 * attempt} s...")
                import time
                time.sleep(10 * attempt)
        if WEIGHTS_SHA256 and _sha256(WEIGHTS_PATH) != WEIGHTS_SHA256:
            os.remove(WEIGHTS_PATH)
            raise RuntimeError(f"The file downloaded from {WEIGHTS_URL} is not the ConSurf-AI weights file "
                               "(checksum mismatch). Please report this issue.")
    elif IN_COLAB:
        from google.colab import files
        print("Please upload the ConSurf-AI weights file (consurf_ai_3B_weights.pth):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No weights file uploaded.")
        with open(WEIGHTS_PATH, "wb") as f:
            f.write(next(iter(uploaded.values())))
    else:
        raise FileNotFoundError(f"Weights not found at {WEIGHTS_PATH} and WEIGHTS_URL is empty.")
    return WEIGHTS_PATH


def download_esm2(max_attempts=4):
    """Fetch the ESM2-3B PyTorch files into the Hugging Face cache (~11 GB; safetensors if the repo has them,
    otherwise the .bin shards; the TensorFlow copies are skipped).
    ESM2 is public, so no Hugging Face token is needed. A token (Colab 🔑 Secrets → HF_TOKEN) only
    raises the rate limit, so the 'unauthenticated requests' warning is hidden when none is set."""
    import time, logging, warnings
    from huggingface_hub import snapshot_download
    warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
    logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
    print(f"Downloading {ESM2_NAME} (~11 GB, first run only)...")
    for attempt in range(1, max_attempts + 1):
        try:
            path = snapshot_download(ESM2_NAME, allow_patterns=["*.json", "*.txt", "*.safetensors"])
            if not any(f.endswith(".safetensors") for f in os.listdir(path)):
                path = snapshot_download(ESM2_NAME, allow_patterns=["*.json", "*.txt", "*.bin"])
            return path
        except Exception as e:                       # network hiccup or HF rate limit (HTTP 429)
            if attempt == max_attempts:
                raise RuntimeError(
                    f"Could not download {ESM2_NAME} from Hugging Face ({type(e).__name__}: {e}).\n"
                    "If this is a rate-limit error (429), create a free token at https://huggingface.co/settings/tokens, "
                    "add it in Colab's 🔑 Secrets panel as HF_TOKEN (enable notebook access), then rerun cell 0."
                ) from None
            wait = 30 * attempt
            print(f"   Download interrupted ({type(e).__name__}); retrying in {wait} s "
                  f"(attempt {attempt + 1}/{max_attempts})...")
            time.sleep(wait)


_MODELS = {}


class _HalfStoredLinear(nn.Module):
    """Linear layer whose weights are stored in float16 (half the memory) but used in float32: on CPUs this gives
    the same results as float32 (unlike bfloat16/int8 computation, which change some grades) and runs at float32 speed."""

    def __init__(self, lin):
        super().__init__()
        self.weight = nn.Parameter(lin.weight.detach().half(), requires_grad=False)
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.detach().clone(), requires_grad=False)

    def forward(self, x):
        return F.linear(x, self.weight.float(), self.bias)


def _store_linear_weights_in_half(model):
    for mod in list(model.modules()):
        for name, child in list(mod.named_children()):
            if isinstance(child, nn.Linear):
                setattr(mod, name, _HalfStoredLinear(child))
    for p in model.parameters():          # small float32 tensors left (embeddings, LayerNorm): copy out of the file map
        if p.dtype == torch.float32:
            p.data = p.data.clone()


def load_models():
    if _MODELS:
        return _MODELS
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
        print("⚠️ No GPU detected: running on the CPU. This works, but it is much slower than on a GPU "
              "(about a second per protein on a GPU). For speed: Runtime → Change runtime type → select a GPU, then run all cells again.")

    ckpt = torch.load(download_consurf_ai_weights(), map_location="cpu", weights_only=True)
    net = AttentionNetwork(ckpt["config"])
    state = {k.removeprefix("_orig_mod.").removeprefix("module."): v for k, v in ckpt["model"].items()}
    net.load_state_dict(state, strict=True)
    net.to(device).eval()

    from transformers import AutoTokenizer, EsmModel
    from transformers.utils import logging as hf_logging
    print("Loading ESM2-3B onto the GPU..." if device.type == "cuda" else "Loading ESM2-3B...")
    dtype = torch.float32
    if device.type == "cuda" and torch.cuda.get_device_properties(0).total_memory / 2**30 < 14:
        dtype = torch.float16                # fp32 ESM2-3B needs ~11.4 GB of weights + activations
    half_store = False
    if device.type == "cpu":
        ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
        # float32 ESM2-3B needs ~11.5 GB of RAM: with less than 16 GB (e.g. Colab's CPU runtime, 12.7 GB) the
        # weights are stored in float16 (~6 GB) and the computation stays in float32 -> same results as float32.
        half_store = ram_gb < 16
    tv = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    kw = {"add_pooling_layer": False, ("dtype" if tv >= (4, 56) else "torch_dtype"): dtype}
    if tv < (5, 0):
        kw["low_cpu_mem_usage"] = True
    # Only the encoder is used (per-residue last-layer embeddings), so the masked-LM head stored in the
    # checkpoint is skipped on purpose; silence transformers' report about those unused weights.
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            tokenizer = AutoTokenizer.from_pretrained(ESM2_NAME)
            try:
                esm = EsmModel.from_pretrained(ESM2_NAME, device_map={"": 0}, **kw) if device.type == "cuda" \
                    else EsmModel.from_pretrained(ESM2_NAME, **kw)
            except (ImportError, ValueError):   # accelerate missing -> plain load
                esm = EsmModel.from_pretrained(ESM2_NAME, **kw).to(device)
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()
    if half_store:
        _store_linear_weights_in_half(esm)
    esm.eval()
    _MODELS.update(device=device, net=net, esm=esm, tokenizer=tokenizer, esm_dtype=dtype)
    print(f"Models ready on {device} (ESM2 in {str(dtype).replace('torch.', '')}"
          + (", weights stored in float16 to fit in memory)." if half_store else ")."))
    return _MODELS


@torch.no_grad()
def predict_grades(sequence):
    """Returns grades (1-9), the 9 grade probabilities, the raw uncertainty (0-8, lower = better),
    the rounded uncertainty and the 'confident' flag (rounded uncertainty < 2), same conventions
    as the evaluation scripts."""
    m = load_models()
    dev = m["device"]
    tokens = m["tokenizer"](sequence, return_tensors="pt").to(dev)
    hidden = m["esm"](**tokens).last_hidden_state[0, 1:1 + len(sequence)].float()   # drop BOS/EOS
    if not torch.isfinite(hidden).all():
        raise RuntimeError("ESM2 produced non-finite embeddings (fp16 overflow?). Try a GPU with more memory.")
    mask = torch.ones(1, len(sequence), dtype=torch.bool, device=dev)
    logits, unc = m["net"](hidden.unsqueeze(0), mask)
    probs = F.softmax(logits[0].float(), dim=-1)
    grades = probs.argmax(dim=-1) + 1
    unc = unc[0].float()
    unc_rounded = torch.floor(unc + 0.5).clamp(0, 8).int()
    return (grades.cpu().numpy(), probs.cpu().numpy(), unc.cpu().numpy(),
            unc_rounded.cpu().numpy(), (unc_rounded < 2).cpu().numpy())


# ── Sequences ────────────────────────────────────────────────────────────────
_ALLOWED_AA = set("ACDEFGHIKLMNPQRSTVWYXUOBZ")
_THREE = {"A": "ALA", "C": "CYS", "D": "ASP", "E": "GLU", "F": "PHE", "G": "GLY", "H": "HIS", "I": "ILE",
          "K": "LYS", "L": "LEU", "M": "MET", "N": "ASN", "P": "PRO", "Q": "GLN", "R": "ARG", "S": "SER",
          "T": "THR", "V": "VAL", "W": "TRP", "Y": "TYR", "U": "SEC", "O": "PYL", "B": "ASX", "Z": "GLX", "X": "UNK"}


def _safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_") or "query"


def parse_fasta_text(text):
    """[(header, raw sequence)]; text without any '>' line is read as one sequence."""
    records, header, seq = [], None, []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith(";"):
            continue
        if line.startswith(">"):
            if header is not None or seq:
                records.append((header, "".join(seq)))
            header, seq = line[1:].strip(), []
        else:
            seq.append(line)
    if header is not None or seq:
        records.append((header, "".join(seq)))
    return records


def clean_sequence(raw):
    """Upper-case, drop spaces/digits/gaps/stops; unknown letters -> X. Returns (sequence, notes)."""
    s = re.sub(r"[\s\d]", "", raw).upper()
    notes = []
    n_gap = len(re.findall(r"[-.*]", s))
    if n_gap:
        s = re.sub(r"[-.*]", "", s)
        notes.append(f"{n_gap} gap/stop character(s) removed")
    bad = sorted(set(s) - _ALLOWED_AA)
    if bad:
        s = "".join(c if c in _ALLOWED_AA else "X" for c in s)
        notes.append(f"unknown character(s) {''.join(bad)} read as X")
    return s, notes


def prepare_sequences(raw_records):
    """List of dicts: id (first word of the header, made unique), description, sequence, status, note."""
    out, used = [], set()
    for i, (header, raw) in enumerate(raw_records, 1):
        header = " ".join((header or "").split()) or f"seq{i}"      # also removes tabs
        base = _safe(header.split()[0])[:60]
        sid, k = base, 2
        while sid in used:
            sid, k = f"{base}_{k}", k + 1
        used.add(sid)
        seq, notes = clean_sequence(raw)
        status = "ok"
        if not seq:
            status = "skipped: empty sequence"
        elif len(seq) > MAX_LEN:
            status = f"skipped: longer than {MAX_LEN} residues"
        out.append({"id": sid, "description": header, "sequence": seq, "status": status, "note": "; ".join(notes)})
    return out


def sequence_grades_table(rec, grades, probs, unc_raw, unc_rounded, confident):
    seq = rec["sequence"]
    df = pd.DataFrame({
        "SEQ_ID": rec["id"],
        "POS": np.arange(1, len(seq) + 1),
        "AA": list(seq),
        "GRADE": grades.astype(int),
        "UNCERTAINTY_RAW": np.round(unc_raw, 3),
        "UNCERTAINTY": unc_rounded.astype(int),
        "CONFIDENT": confident.astype(bool),
    })
    for g in range(9):
        df[f"P_GRADE_{g + 1}"] = np.round(probs[:, g], 4)
    return df


def sequence_summary_row(rec, df=None):
    row = {"SEQ_ID": rec["id"], "DESCRIPTION": rec["description"], "LENGTH": len(rec["sequence"]),
           "STATUS": rec["status"], "NOTE": rec["note"]}
    if df is not None:
        row.update({"N_UNCERTAIN": int((~df.CONFIDENT).sum()),
                    "FRACTION_UNCERTAIN": round(float((~df.CONFIDENT).mean()), 3),
                    "MEAN_GRADE": round(float(df.GRADE.mean()), 2)})
        counts = df.GRADE.value_counts()
        row.update({f"N_GRADE_{g}": int(counts.get(g, 0)) for g in range(1, 10)})
    row["SEQUENCE"] = rec["sequence"]
    return row


def summary_table(records, tables):
    df = pd.DataFrame([sequence_summary_row(r, tables.get(r["id"])) for r in records])
    for c in [c for c in df if c.startswith("N_")]:      # whole numbers (empty for skipped sequences)
        df[c] = df[c].astype("Int64")
    return df


def sequence_view_df(df):
    """Columns expected by sequence_html (numbering = position in the sequence)."""
    return df.assign(CHAIN="", RESIDUE_NUMBER=df.POS, RESNAME=df.AA.map(_THREE).fillna("UNK"))


# ── Result files: tab-separated tables + README.txt ─────────────────────────
# Tables are written without comment lines (they open cleanly in Excel, R and pandas); the description of the
# run and of every column goes to README.txt next to them.
COLUMN_HELP = {
    "SEQ_ID": "sequence identifier (first word of the FASTA header, made unique)",
    "POS": "position in the sequence (1 = first residue)",
    "CHAIN": "chain ID in the structure file",
    "RESIDUE_NUMBER": "residue number in the structure file (with insertion code, if any)",
    "RESNAME": "residue name (three-letter code)",
    "AA": "amino acid (one-letter code)",
    "GRADE": "ConSurf-AI conservation grade: 1 = most variable ... 9 = most conserved",
    "UNCERTAINTY_RAW": "predicted uncertainty of the grade, 0-8 (lower = better)",
    "UNCERTAINTY": "UNCERTAINTY_RAW rounded to the nearest integer",
    "CONFIDENT": "True if UNCERTAINTY < 2; False = uncertain prediction (yellow in the viewer; same role as "
                 "ConSurf's 'insufficient data' flag)",
    "P_GRADE_1 ... P_GRADE_9": "predicted probability of each grade (they sum to 1)",
}


def write_readme(out_dir, title, input_desc, files, column_sets):
    """files: [(name, description)]; column_sets: [(table name, [columns])]."""
    w = max(len(n) for n, _ in files) + 2
    out = [title, "=" * len(title), "",
           f"Input:  {input_desc}",
           f"Date:   {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M} UTC",
           "Model:  ConSurf-AI (attention network on ESM2-3B embeddings)", "",
           "Files", "-----"]
    out += [f"{n:<{w}}{d}" for n, d in files]
    for table, cols in column_sets:
        out += ["", f"Columns of {table}", "-" * (11 + len(table))]
        shown = [c for c in cols if not c.startswith("P_GRADE_")]
        if any(c.startswith("P_GRADE_") for c in cols):
            shown.append("P_GRADE_1 ... P_GRADE_9")
        cw = max(len(c) for c in shown) + 2
        out += [f"{c:<{cw}}{COLUMN_HELP.get(c, '')}" for c in shown]
    out += ["", "Tables are tab-separated (.tsv). Excel: File > Open (or drag the file onto Excel). "
                "R: read.delim(file). Python: pandas.read_csv(file, sep='\\t')."]
    path = os.path.join(out_dir, "README.txt")
    with open(path, "w") as f:
        f.write("\n".join(out) + "\n")
    return path
COLUMN_HELP.update({
    "DESCRIPTION": "full FASTA header (or the name given in cell 1)",
    "LENGTH": "number of residues after cleaning",
    "STATUS": "ok, skipped (empty or longer than 2000 residues) or error",
    "NOTE": "changes made to the sequence (removed gaps/stops, unknown letters read as X)",
    "N_UNCERTAIN": "number of residues with an uncertain prediction (CONFIDENT = False)",
    "FRACTION_UNCERTAIN": "N_UNCERTAIN / LENGTH",
    "MEAN_GRADE": "mean grade over the sequence",
    "N_GRADE_1 ... N_GRADE_9": "number of residues with each grade",
    "SEQUENCE": "the sequence as used for the prediction",
})


def write_sequence_outputs(job, source, records, tables, summary):
    out_dir = os.path.join(WORK_DIR, "results", job)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(os.path.join(out_dir, "per_sequence"))
    files = []
    p = os.path.join(out_dir, f"{job}_ConSurf-AI_summary.tsv")
    summary.to_csv(p, sep="\t", index=False)
    files.append((p, "one row per input sequence: status, length, uncertain residues, grade counts, sequence"))
    all_cols = []
    if tables:
        all_res = pd.concat(tables.values(), ignore_index=True)
        all_cols = list(all_res.columns)
        p = os.path.join(out_dir, f"{job}_ConSurf-AI_all_residues.tsv")
        all_res.to_csv(p, sep="\t", index=False)
        files.append((p, "one row per residue of every sequence (grade, uncertainty, probabilities)"))
        p = os.path.join(out_dir, f"{job}_ConSurf-AI_grade_strings.fasta")
        with open(p, "w") as f:
            for sid, df in tables.items():
                f.write(f">{sid} sequence\n{''.join(df.AA)}\n>{sid} grades\n{''.join(map(str, df.GRADE))}\n"
                        f">{sid} uncertainty\n{''.join(map(str, df.UNCERTAINTY))}\n")
        files.append((p, "per sequence: the sequence, its grades and its rounded uncertainties as aligned strings "
                         "(one character per residue; uncertainty >= 2 = uncertain)"))
    for sid, df in tables.items():
        df.to_csv(os.path.join(out_dir, "per_sequence", f"{sid}_ConSurf-AI_grades.tsv"), sep="\t", index=False)
    if tables:
        files.append(("per_sequence/", "the per-residue table, one file per sequence (<SEQ_ID>_ConSurf-AI_grades.tsv)"))
    sum_cols = [c for c in summary.columns if not c.startswith("N_GRADE_")]
    if any(c.startswith("N_GRADE_") for c in summary.columns):
        sum_cols.insert(sum_cols.index("SEQUENCE"), "N_GRADE_1 ... N_GRADE_9")
    readme = write_readme(out_dir, "ConSurf-AI results (sequences)", f"{source}, {len(records)} sequence(s)",
                          [(os.path.basename(p.rstrip("/")) + ("/" if p.endswith("/") else ""), d) for p, d in files],
                          [("the summary table", sum_cols)] + ([("the residue tables", all_cols)] if all_cols else []))
    return out_dir, [p for p, _ in files if not p.endswith("/")] + [readme]


def legend_html():
    boxes = "".join(
        f'<div style="width:30px;height:22px;background:{CONSURF_COLORS[g]};border:1px solid #888;'
        f'display:flex;align-items:center;justify-content:center;color:{"#fff" if g == 9 else "#000"};'
        f'font:bold 12px sans-serif">{g}</div>' for g in range(1, 10))
    return (f'<div style="display:flex;align-items:center;gap:8px;font:13px sans-serif;margin:6px 0">'
            f'<span>Variable</span><div style="display:flex">{boxes}</div><span>Conserved</span>'
            f'<div style="width:30px;height:22px;background:{LOWCONF_COLOR};border:1px solid #888;margin-left:16px"></div>'
            f'<span>Uncertain (uncertainty ≥ 2)</span></div>')


def sequence_html(df, highlight_low_confidence=True, per_row=50):
    rows = []
    for start in range(0, len(df), per_row):
        chunk = df.iloc[start:start + per_row]
        cells = []
        for i, r in enumerate(chunk.itertuples()):
            low = highlight_low_confidence and not r.CONFIDENT
            bg = LOWCONF_COLOR if low else CONSURF_COLORS[r.GRADE]
            fg = "#fff" if (r.GRADE == 9 and not low) else "#000"
            tip = f"{r.RESNAME} {r.CHAIN}{r.RESIDUE_NUMBER} – grade {r.GRADE}, uncertainty {r.UNCERTAINTY_RAW:.2f}"
            sep = "margin-left:5px;" if i and i % 10 == 0 else ""
            cells.append(f'<span title="{_html.escape(tip)}" style="{sep}display:inline-block;width:15px;'
                         f'text-align:center;background:{bg};color:{fg};font:13px monospace;line-height:16px">'
                         f'{r.AA}<br><span style="font-size:9px">{r.GRADE}</span></span>')
        rows.append(f'<div style="white-space:nowrap;margin:2px 0"><span style="display:inline-block;width:52px;'
                    f'text-align:right;margin-right:6px;font:11px monospace;color:#888">{chunk.RESIDUE_NUMBER.iloc[0]}'
                    f'</span>{"".join(cells)}</div>')
    return f'<div style="overflow-x:auto;padding:4px 0;background:#fff">{"".join(rows)}</div>'


SEQ_STATE = {}
CONSURF_AI_READY = True
print(f"✅ ConSurf-AI helpers ready (torch {torch.__version__}, transformers {transformers.__version__}, "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none – will run on the CPU (slower)'})")

# ── Download model weights now so later cells only load them ────────────────
download_consurf_ai_weights()
print("✅ ConSurf-AI weights ready")
download_esm2()
print("✅ ESM2-3B weights ready")


In [ ]:
#@title 1. Provide sequence(s) {display-mode: "form"}
#@markdown Paste **one protein sequence** (one-letter code; spaces, digits and gaps are ignored), **or** tick **upload_fasta** to upload a FASTA file with one or many sequences (`.fasta`, `.fa`, `.faa`, `.txt`, optionally `.gz`).
sequence = "VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"  #@param {type:"string"}
sequence_name = "HBA_HUMAN"  #@param {type:"string"}
upload_fasta = False  #@param {type:"boolean"}
#@markdown `job_name` is used for the output folder and file names.
job_name = "my_sequences"  #@param {type:"string"}

if "CONSURF_AI_READY" not in globals():
    raise RuntimeError("Please run cell 0 first.")

if upload_fasta:
    if not IN_COLAB:
        raise RuntimeError("Uploading is only available in Google Colab.")
    from google.colab import files
    _uploaded = files.upload()
    if not _uploaded:
        raise RuntimeError("No file uploaded.")
    _fname, _data = next(iter(_uploaded.items()))
    if _fname.lower().endswith(".gz"):
        import gzip
        _data = gzip.decompress(_data)
    _raw = parse_fasta_text(_data.decode("utf-8", errors="replace"))
    _source = f"uploaded file {_fname}"
else:
    if not sequence.strip():
        raise ValueError("Paste a sequence, or tick 'upload_fasta' to upload a FASTA file.")
    _raw = [(sequence_name.strip() or "query", sequence)]
    _source = "pasted sequence"
_records = prepare_sequences(_raw)
if not _records:
    raise ValueError("No sequence found in the input.")
SEQ_STATE = {"records": _records, "job": _safe(job_name), "source": _source}

_ok = [r for r in _records if r["status"] == "ok"]
print(f"Loaded {len(_records)} sequence(s) from {_source}: {len(_ok)} ready, {len(_records) - len(_ok)} skipped.")
if _ok:
    _lens = [len(r["sequence"]) for r in _ok]
    print(f"Lengths: {min(_lens)}–{max(_lens)} residues (total {sum(_lens):,}). "
          f"Estimated run time on a T4: ~{max(1, round(len(_ok) * 0.5 / 60))} min after ESM2 is loaded.")
_show = pd.DataFrame([{"id": r["id"], "length": len(r["sequence"]), "status": r["status"], "note": r["note"],
                       "sequence": r["sequence"] if len(r["sequence"]) <= 50 else r["sequence"][:47] + "..."}
                      for r in _records]).set_index("id")
display(_show.head(50))
if len(_records) > 50:
    print(f"... and {len(_records) - 50} more.")


In [ ]:
#@title 2. Run ConSurf-AI {display-mode: "form"}
#@markdown Predicts every sequence and writes the result tables. The first run loads ESM2-3B onto the GPU (~1 min).
#@markdown The coloured sequences of the first `show_max` sequences are shown below (all of them are in the files).
show_max = 10  #@param {type:"integer"}
highlight_low_confidence = True  #@param {type:"boolean"}
import time

if not SEQ_STATE.get("records"):
    raise RuntimeError("Please run cell 1 first.")
load_models()
_recs = SEQ_STATE["records"]
_todo = [r for r in _recs if r["status"] == "ok"]
_tables, _t0, _last = {}, time.time(), time.time()
for _i, _r in enumerate(_todo, 1):
    try:
        _tables[_r["id"]] = sequence_grades_table(_r, *predict_grades(_r["sequence"]))
    except Exception as _e:                      # keep going (e.g. GPU out of memory on a long sequence)
        _r["status"] = f"error: {type(_e).__name__}: {str(_e)[:150]}"
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if time.time() - _last > 10 or _i == len(_todo):
        _last = time.time()
        _el = _last - _t0
        print(f"   {_i}/{len(_todo)} sequences done ({_el:.0f} s"
              + (f", ~{_el / _i * (len(_todo) - _i):.0f} s left)" if _i < len(_todo) else ")"))
_summary = summary_table(_recs, _tables)
_out_dir, _written = write_sequence_outputs(SEQ_STATE["job"], SEQ_STATE["source"], _recs, _tables, _summary)
SEQ_STATE.update(tables=_tables, summary=_summary, out_dir=_out_dir, files=_written)

_failed = [r for r in _recs if r["status"] != "ok"]
print(f"\n✅ Predicted {len(_tables)} sequence(s) ({sum(len(t) for t in _tables.values()):,} residues) "
      f"in {time.time() - _t0:.1f} s" + (f"; {len(_failed)} skipped or failed (see STATUS)." if _failed else "."))
print(f"Files written to {_out_dir}:")
for _f in _written:
    print("   " + os.path.basename(_f))
if _tables:
    print(f"   per_sequence/ ({len(_tables)} files)")
_cols = [c for c in ["SEQ_ID", "LENGTH", "STATUS", "N_UNCERTAIN", "FRACTION_UNCERTAIN", "MEAN_GRADE"] if c in _summary]
display(_summary[_cols].set_index("SEQ_ID").head(50))

if _tables and show_max > 0:
    display(HTML(legend_html()))
    for _sid, _df in list(_tables.items())[:show_max]:
        _desc = next(r["description"] for r in _recs if r["id"] == _sid)
        display(HTML(f"<h4 style='font-family:sans-serif;margin:10px 0 2px'>{_html.escape(_desc)} "
                     f"<span style='font-weight:normal;color:#888'>({len(_df)} residues, "
                     f"{(~_df.CONFIDENT).sum()} uncertain)</span></h4>"
                     + sequence_html(sequence_view_df(_df), highlight_low_confidence)))
    if len(_tables) > show_max:
        print(f"(Coloured view shown for the first {show_max} of {len(_tables)} sequences.)")


In [ ]:
#@title 3. Download results {display-mode: "form"}
#@markdown Downloads a zip with the summary table, the table of all residues, one table per sequence, the aligned grade strings and a README.

if "out_dir" not in SEQ_STATE:
    raise RuntimeError("Please run cell 2 first.")
_zip_path = os.path.join(WORK_DIR, f"{SEQ_STATE['job']}_ConSurf-AI.zip")
with zipfile.ZipFile(_zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for _root, _, _fs in os.walk(SEQ_STATE["out_dir"]):
        for _f in sorted(_fs):
            _p = os.path.join(_root, _f)
            z.write(_p, arcname=os.path.relpath(_p, SEQ_STATE["out_dir"]))
print(f"Created {_zip_path}")
if IN_COLAB:
    from google.colab import files
    files.download(_zip_path)


### Output files
| File | Content |
|---|---|
| `*_summary.tsv` | One row per input sequence: ID, FASTA header, length, status (ok / skipped / error), changes made to the sequence, number and fraction of uncertain residues, mean grade, number of residues in each grade, sequence |
| `*_all_residues.tsv` | One row per residue of every sequence: sequence ID, position, amino acid, **grade (1–9)**, **uncertainty** (raw 0–8 value and rounded; lower = better), confidence flag (rounded uncertainty < 2) and the probability of each of the 9 grades |
| `per_sequence/*_grades.tsv` | The same per-residue table, one file per sequence |
| `*_grade_strings.fasta` | For each sequence: the sequence, its grades and its rounded uncertainties as three aligned FASTA records (one character per residue) |
| `README.txt` | Description of the run (input, date) and of every column |

**Notes.** Sequences of up to 2000 residues are supported; longer ones are listed as skipped in the summary. Gaps (`-`, `.`) and stop codons (`*`) are removed and unknown letters are read as `X`. Sequence IDs are taken from the first word of each FASTA header (made unique if repeated).